# Analyze 🤗 HuggingFace audio dataset

This notebook will show how to use DataChain for audio data, including how to:

- [Access audio data from the 🤗 HuggingFace storage.](#ingesting)
- [Apply emotion analysis model.](#emotions)
- [Apply Whisper speech recognition model to get transcripts.](#transcripts)
- [Join those data into a coherent dataset.](#joining)
- [Save the chain results as a dataset](#saving)
- [Filter and inspect the data.](#filter)

<a id='setup'></a>

## Setup

To start, install the dependencies and import the packages needed.

In [1]:
%pip install -q  ipywidgets "datachain[torch]" transformers hf_transfer


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [54]:
import torch
from transformers import pipeline, Pipeline

import datachain as dc
from datachain import C, File, DataModel
from datachain.func import path

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'


<a id='ingesting'></a>

## Get data and create DataChain

We will use a dataset from https://huggingface.co/datasets/fsicoli/common_voice_22_0. This mirror keeps the audio in `tar` shards on Hugging Face, which makes it a good fit for demonstrating DataChain's file-based workflow.

In the repository layout, `en` is the ISO language code for English. We will first look at `hf://datasets/fsicoli/common_voice_22_0/audio/en` so you can see several split tar files, and then we will process the lighter `dev` split from `audio/en/dev` together with the matching transcript metadata from `transcript/en/dev.tsv` to keep the example runnable.

We can create a DataChain from the data source. DataChain can create a single dataset from many files in a storage bucket, folder, or other path. DataChain will generate one record per file, with each record including a `file` signal that saves metadata needed to read the file. In that way, DataChain keeps a link to the original storage without having to copy the files or load all file contents in memory.

### Indexing storage

To create a chain from a directory of files, use `dc.read_storage()` and point to the relevant subdirectory. Here we first index the broader English audio directory so the preview shows a few tar files for different splits, while the processing steps later in the notebook use the lighter `dev` split.

In [55]:
dc.read_storage("hf://datasets/fsicoli/common_voice_22_0/audio/en").show(5)

,file,file
,path,size
0,fsicoli/common_voice_22_0/audio/en/dev/en_dev_...,712545792
1,fsicoli/common_voice_22_0/audio/en/test/en_tes...,706425856
2,fsicoli/common_voice_22_0/audio/en/other/en_ot...,1318713344
3,fsicoli/common_voice_22_0/audio/en/other/en_ot...,1363123712
4,fsicoli/common_voice_22_0/audio/en/other/en_ot...,1259754496



[Limited by 5 rows]


DataChain created a record for each file in that directory, generating a `file` signal for each file. The `file` signal contains subsignals with metadata about each file, like `file.name` and `file.size`. You can define your own aggregate signals like this using Pydantic models, although that is outside the scope of this tutorial. You can also use the `file` signal to open and read the file as needed. We will come back to this later in the notebook.

### Basic filtering and TAR processing

The audio lives in `tar` archives, while the transcript metadata lives in a TSV file. We expand the `audio/en/dev` tar shard with `process_tar`, read the transcript TSV with `dc.read_csv(..., delimiter="\t")`, and merge the two DataChains by filename.

In [ ]:
from datachain.lib.tar import process_tar

transcript_dc = (
    dc.read_csv(
        "hf://datasets/fsicoli/common_voice_22_0/transcript/en/dev.tsv",
        delimiter="\t",
        source=False,
        # The English TSV has messy values in sentence_domain, so pin it to string.
        column_types={"sentence_domain": "string"},
    )
    .persist()
)

repo_dc = (
    dc.read_storage("hf://datasets/fsicoli/common_voice_22_0/audio/en/dev")
        .settings(cache=True)
        .gen(file=process_tar)
        .mutate(name=path.name(C("file.path")))
        .merge(transcript_dc, on="name", right_on="path", inner=True)
        .mutate(reference_text=C("sentence"))
        .persist()
)

Preparing: 0 rows [00:00, ? rows/s]

Generated: 0 rows [00:00, ? rows/s]

Processed: 0 rows [00:00, ? rows/s]

Preparing: 0 rows [00:00, ? rows/s]

Generated: 0 rows [00:00, ? rows/s]

...

Generated: 0 rows [00:00, ? rows/s]

Processed: 0 rows [00:00, ? rows/s]

Preparing: 0 rows [00:00, ? rows/s]

Generated: 0 rows [00:00, ? rows/s]

Processed: 0 rows [00:00, ? rows/s]

Let's preview the data. We now have `mp3` file entries from inside the tar shard together with the reference transcript for each clip. Each audio file is still a virtual reference inside the original archive, so no copying is happening.

In [68]:
repo_dc.select("name", "reference_text").show(3)

,name,reference_text
0,common_voice_en_100546.mp3,Sarah told him that she was there to see her b...
1,common_voice_en_100548.mp3,Galileo Galilei was the first man who observed...
2,common_voice_en_100738.mp3,"The boy swore that, every time he heard the al..."



[Limited by 3 rows]


<a id='emotions'></a>

## Emotions classification

Now we can apply a model from Hugging Face to classify emotions in each individual `mp3` recording. We are running a DataChain `map` operation that applies the custom `emotions()` function and returns a `pydantic` model `Emotions` with the results. Each result is automatically saved as a set of additional columns in the DataChain table for each recording.

In [71]:
class Emotions(DataModel):
  anxiety: float = 0.0
  boredom: float = 0.0
  neutral: float = 0.0
  panic: float = 0.0
  sadness: float = 0.0
  shame: float = 0.0
  cold_anger: float = 0.0
  contempt: float = 0.0
  despair: float = 0.0
  disgust: float = 0.0
  elation: float = 0.0
  happy: float = 0.0
  hot_anger: float = 0.0
  interest: float = 0.0


def emotions(helper: Pipeline, file: File) -> Emotions:
  # `helper` is the Hugging Face pipeline object
  result = helper(file.read())
  return Emotions(
    **{emotion["label"].replace("-", "_"): emotion["score"] for emotion in result}
  )


emotions_dc = (
    repo_dc
       # Put `.settings(parallel=4)` on a bigger machine
  	  .limit(10)
      .setup(helper=lambda: pipeline(model="Krithika-p/my_awesome_emotions_model", device=device))
  	  .map(emotions=emotions)
      .persist()
)


Preparing: 0 rows [00:00, ? rows/s]

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

Processed: 0 rows [00:00, ? rows/s]

Let's preview the emotions classifier results:

In [59]:
emotions_dc.select("name", "emotions").show(3)

,name,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions
,,anxiety,boredom,neutral,panic,sadness,shame,cold_anger,contempt,despair,disgust,elation,happy,hot_anger,interest
0,common_voice_en_100229.mp3,0.019159,0.326695,0.027214,0.007535,0.095398,0.143197,0.033843,0.170145,0.024168,0.037878,0.007724,0.01585,0.011981,0.014985
1,common_voice_en_100538.mp3,0.025511,0.036973,0.0137,0.022107,0.020106,0.021752,0.152584,0.263877,0.023112,0.130538,0.026666,0.082731,0.016636,0.043819
2,common_voice_en_100539.mp3,0.035417,0.045819,0.01017,0.020052,0.043227,0.031632,0.071549,0.133608,0.061939,0.257236,0.042794,0.082355,0.017063,0.026349



[Limited by 3 rows]


<a id='transcripts'></a>

## Getting transcriptions

Similar to the emotion classification above, we run another model to get transcriptions. Because this dataset already includes reference text, we can compare the model output with the original sentence from the dataset.

In [69]:
def text(helper: Pipeline, file: File) -> str:
  result = helper(file.read())
  return result["text"]


text_dc = (
    repo_dc
      # Put `.settings(parallel=4)` on a bigger machine
      .limit(10)
      # Make it large-v3 for better accuracy if you have a GPU / bigger machine 
      .setup(helper=lambda: pipeline(
          "automatic-speech-recognition",
          model="openai/whisper-tiny",
          dtype=torch.float32,
          device=device,
          generate_kwargs={"language": "en", "task": "transcribe"},
      ))
  	  .map(text=text)
      .persist()
)


Preparing: 0 rows [00:00, ? rows/s]

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Processed: 0 rows [00:00, ? rows/s]

Let's preview the transcripts results:

In [70]:
text_dc.select("name", "reference_text", "text").show(3)

,name,reference_text,text
0,common_voice_en_100546.mp3,Sarah told him that she was there to see her b...,They're all told him that she was there to se...
1,common_voice_en_100548.mp3,Galileo Galilei was the first man who observed...,Galileo Galileo was the first man who observe...
2,common_voice_en_100738.mp3,"The boy swore that, every time he heard the al...",The boy saw that every time he held the alarm...



[Limited by 3 rows]


<a id='joining'></a>

## Merging the results into a single dataset

The emotion and transcription chains can be merged into a single dataset while keeping the reference transcript from the original metadata:

In [72]:
merged_dc = text_dc.merge(emotions_dc, on=C("file.path"), inner=True)
merged_dc.select("name", "reference_text", "text", "emotions").show(3)

,name,reference_text,text,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions
,,,,anxiety,boredom,neutral,panic,sadness,shame,cold_anger,contempt,despair,disgust,elation,happy,hot_anger,interest
0,common_voice_en_100546.mp3,Sarah told him that she was there to see her b...,They're all told him that she was there to se...,0.027589,0.096766,0.015795,0.010786,0.041981,0.051325,0.085954,0.329168,0.025919,0.116885,0.012722,0.034937,0.011712,0.030675
1,common_voice_en_100548.mp3,Galileo Galilei was the first man who observed...,Galileo Galileo was the first man who observe...,0.030064,0.071195,0.013716,0.014903,0.037572,0.041062,0.096162,0.280003,0.030888,0.166556,0.018731,0.046153,0.012898,0.029982
2,common_voice_en_100738.mp3,"The boy swore that, every time he heard the al...",The boy saw that every time he held the alarm...,0.047941,0.19377,0.024622,0.009125,0.089421,0.118589,0.040163,0.163536,0.058423,0.072999,0.01211,0.031245,0.011628,0.034795



[Limited by 3 rows]


<a id='saving'></a>

## Saving the chain

The result is saved using `DataChain.save()`. If name is provided this will save a persistent, versioned dataset that we can recover anytime in the future. It also will prevent DataChain from recomputing the same steps when running the chain again. If name is not provided it has an effect only on the current session to avoid recomputing certain data chains again and again, but it's not persisted.


In [73]:
merged_dc.save("common-voice-22-en")
dc.read_dataset("common-voice-22-en").show(3)

,file,file,name,client_id,path,sentence_id,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment,reference_text,text,right_file,right_file,right_name,right_client_id,right_path,right_sentence_id,right_sentence_domain,right_up_votes,right_down_votes,right_age,right_gender,right_accents,right_variant,right_locale,right_segment,right_reference_text,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions,emotions
,path,size,,,,,,,,,,,,,,,,path,size,,,,,,,,,,,,,,,anxiety,boredom,neutral,panic,sadness,shame,cold_anger,contempt,despair,disgust,elation,happy,hot_anger,interest
0,fsicoli/common_voice_22_0/audio/en/dev/en_dev_...,44961,common_voice_en_100546.mp3,ec2cd8203e261fecb7a08fe7082464c17ab22735e69df9...,common_voice_en_100546.mp3,036a10312816b365eadbd31df6bcd24be82157c1e758ee...,None,3,2,None,None,None,None,en,None,Sarah told him that she was there to see her b...,They're all told him that she was there to se...,fsicoli/common_voice_22_0/audio/en/dev/en_dev_...,44961,common_voice_en_100546.mp3,ec2cd8203e261fecb7a08fe7082464c17ab22735e69df9...,common_voice_en_100546.mp3,036a10312816b365eadbd31df6bcd24be82157c1e758ee...,None,3,2,None,None,None,None,en,None,Sarah told him that she was there to see her b...,0.027589,0.096766,0.015795,0.010786,0.041981,0.051325,0.085954,0.329168,0.025919,0.116885,0.012722,0.034937,0.011712,0.030675
1,fsicoli/common_voice_22_0/audio/en/dev/en_dev_...,56481,common_voice_en_100548.mp3,ec2cd8203e261fecb7a08fe7082464c17ab22735e69df9...,common_voice_en_100548.mp3,a6a6ce1d0016ebe0ad490eb707ff645496e298d0547b3c...,None,2,0,None,None,None,None,en,None,Galileo Galilei was the first man who observed...,Galileo Galileo was the first man who observe...,fsicoli/common_voice_22_0/audio/en/dev/en_dev_...,56481,common_voice_en_100548.mp3,ec2cd8203e261fecb7a08fe7082464c17ab22735e69df9...,common_voice_en_100548.mp3,a6a6ce1d0016ebe0ad490eb707ff645496e298d0547b3c...,None,2,0,None,None,None,None,en,None,Galileo Galilei was the first man who observed...,0.030064,0.071195,0.013716,0.014903,0.037572,0.041062,0.096162,0.280003,0.030888,0.166556,0.018731,0.046153,0.012898,0.029982
2,fsicoli/common_voice_22_0/audio/en/dev/en_dev_...,33057,common_voice_en_100738.mp3,18ea45294f98a4b197e502865f674f03c7e59842aa3a74...,common_voice_en_100738.mp3,29b8c698c8df4b60e6fad42bf82b0620ed162564a9eca9...,None,3,1,thirties,male_masculine,"West Indies and Bermuda (Bahamas, Bermuda, Jam...",None,en,None,"The boy swore that, every time he heard the al...",The boy saw that every time he held the alarm...,fsicoli/common_voice_22_0/audio/en/dev/en_dev_...,33057,common_voice_en_100738.mp3,18ea45294f98a4b197e502865f674f03c7e59842aa3a74...,common_voice_en_100738.mp3,29b8c698c8df4b60e6fad42bf82b0620ed162564a9eca9...,None,3,1,thirties,male_masculine,"West Indies and Bermuda (Bahamas, Bermuda, Jam...",None,en,None,"The boy swore that, every time he heard the al...",0.047941,0.19377,0.024622,0.009125,0.089421,0.118589,0.040163,0.163536,0.058423,0.072999,0.01211,0.031245,0.011628,0.034795



[Limited by 3 rows]


<a id='filter'></a>

## Inspecting results

Let's take one sample from the saved dataset and inspect the signals it contains.

In [74]:
dc_chain = dc.read_dataset("common-voice-22-en")
sample = dc_chain.limit(1)

We can use `DataChain.to_iter()` to extract values from the sample. Here's an example of collecting a subset of the signals from that row:

In [75]:
sample_results = list(sample.to_iter("file", "name", "reference_text", "text"))
sample_results

[(File(source='hf://datasets', path='fsicoli/common_voice_22_0/audio/en/dev/en_dev_0.tar/en_dev_0/common_voice_en_100546.mp3', size=44961, version='7e5a529dd3afca40bf56c39598b642367fe101bf', etag='8abcf32dc395a5f071234d135ecce03e', is_latest=True, last_modified=datetime.datetime(1970, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), location=[{'vtype': 'tar', 'parent': {'source': 'hf://datasets', 'path': 'fsicoli/common_voice_22_0/audio/en/dev/en_dev_0.tar', 'size': 712545792, 'version': '7e5a529dd3afca40bf56c39598b642367fe101bf', 'etag': '78250e23da9f9db6a39e094f51b53e3851803099', 'is_latest': True, 'last_modified': '2025-06-30 10:38:48+00:00', 'location': None}, 'size': 44961, 'offset': 1024}]),
  'common_voice_en_100546.mp3',
  'Sarah told him that she was there to see her brother.',
  " They're all told him that she was there to see her brother.")]

The example has an output for each signal:
- `file` returns a special `File` object.
- `name` returns a short file name.
- `reference_text` returns the original transcript from the dataset metadata.
- `text` returns the model transcript.

Let's now render and listen to this example:

In [76]:
import IPython

file = next(sample.to_iter("file"))[0]
IPython.display.Audio(file.read())